# 05 — Executive Recommendations

Synthesize top insights, prioritized actions, and the **$100K+** impact case.

**Core logic:** `src/impact_model.py`, `reports/executive_memo.md`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from config import IMPACT_DEFAULTS, measurement_plan

TABLES = ROOT / "outputs" / "tables"

## Top insights

Three findings from theme, driver, and store opportunity analysis.

In [ ]:
theme_summary = pd.read_csv(TABLES / "theme_summary.csv")
theme_impact = pd.read_csv(TABLES / "theme_impact.csv")
drivers = pd.read_csv(TABLES / "driver_importance.csv")
stores = pd.read_csv(TABLES / "store_opportunity_ranking.csv")

top_vol = theme_summary.sort_values("comment_count", ascending=False).iloc[0]
worst_nps_gap = theme_impact.sort_values("theme_nps_gap").iloc[0]
top_driver = drivers.sort_values("rank").iloc[0]
top_store = stores.iloc[0]

insights = pd.DataFrame([
    {
        "insight": "Most frequent ≠ most damaging",
        "detail": f"{top_vol['primary_theme']} has highest volume ({top_vol['share_of_comments']:.1%}), "
                  f"but {worst_nps_gap['primary_theme']} has worst NPS gap ({worst_nps_gap['theme_nps_gap']:.1f}).",
    },
    {
        "insight": "Largest loyalty driver",
        "detail": f"{top_driver['driver']} (odds ratio {top_driver['odds_ratio']:.2f}).",
    },
    {
        "insight": "Highest-value store opportunity",
        "detail": f"{top_store['store_name']} — score {top_store['opportunity_score']}, "
                  f"theme {top_store['top_negative_theme']}.",
    },
])
insights

## Recommended actions

Cross-functional priorities from `store_opportunity_ranking.csv` and segment analysis.

In [ ]:
actions = [
    "1. Improve speed of service in priority drive-thru stores (peak-hour staffing, queue management).",
    "2. Standardize drink consistency in low-NPS markets (recipe audits, barista retraining).",
    "3. Improve mobile order pickup reliability for mobile-first guests (app-to-store handoff).",
]
for a in actions:
    print(a)

print("\nTop store actions from ranking:")
stores.head(5)[["store_name", "store_type", "top_negative_theme", "recommended_action"]]

## Impact estimate

Default sizing from `impact_summary.csv` (`src/impact_model.py`):

```
incremental_revenue = target_stores × avg_daily_transactions × avg_ticket
                    × window_days × repeat_visit_lift
```

In [ ]:
impact = pd.read_csv(TABLES / "impact_summary.csv")
snapshot = json.loads((TABLES / "impact_sizing.json").read_text())
row = impact.iloc[0]

print("=== Default impact case ===")
print(f"Initiative: {row['initiative']}")
print(f"Target stores: {int(row['target_store_count'])}")
print(f"Avg daily transactions: {row['avg_daily_transactions']}")
print(f"Avg ticket: ${row['avg_ticket']:.2f}")
print(f"Window: {int(row['improvement_window_days'])} days")
print(f"Repeat-visit lift: {float(row['expected_repeat_visit_lift']):.1%}")
print(f"Estimated incremental revenue: ${row['estimated_incremental_revenue']:,.0f}")
print(f"Meets $100K threshold: {snapshot['meets_100k_threshold']}")
print(f"\nAssumptions: {row['assumptions']}")

In [ ]:
print("=== Measurement plan ===")
print(measurement_plan(IMPACT_DEFAULTS["improvement_window_days"]))

## Executive memo

Full narrative in `reports/executive_memo.md`.

In [ ]:
memo_path = ROOT / "reports" / "executive_memo.md"
if memo_path.exists():
    print(memo_path.read_text()[:2500])
    print("\n... [truncated — open reports/executive_memo.md for full memo] ...")

## Next steps

1. Launch a 90-day pilot in top 30 opportunity stores with matched controls.
2. Track before/after metrics per `measurement_plan()` in `src/config.py`.
3. Re-run `make build` monthly to refresh rankings as new survey waves arrive.